# QFF2026 QML Track — PS-Q2 reproducible workflow

Uses seed 42. Cached ideal-kernel tables are loaded by default; set `RECOMPUTE_IDEAL_ABLATION = True` to recompute the full ideal grid. Quantum noise/Task A/Task B require the pinned Qiskit + Aer environment. This notebook records dependency blockers rather than substituting invented results.

In [1]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
RECOMPUTE_IDEAL_ABLATION = False
SEED = 42
print('Root:', ROOT, '| seed:', SEED)

Root: C:\MyChromeDownloads\QFF2026_QML_Submission_UPDATED\QFF2026_QML_Submission | seed: 42


## 1. Data preparation

In [2]:
from src.data import load_q2_data
X_train, y_train, X_test, sample_ids, label_to_int, int_to_label = load_q2_data(ROOT)
print('Train shape:', X_train.shape, '| Test feature shape:', X_test.shape)
print('Label mapping:', label_to_int, '| Train label counts:', y_train.value_counts().to_dict())
assert 'sample_id' not in X_test.columns and len(sample_ids) == 200

Train shape: (800, 50) | Test feature shape: (200, 50)
Label mapping: {'A': 0, 'B': 1} | Train label counts: {1: 414, 0: 386}


## 2. Classical baselines (cached Prompt 2 results)

In [3]:
classical_path = ROOT / 'results/tables/classical_baselines.csv'
classical = pd.read_csv(classical_path)
display(classical[['model','balanced_accuracy_mean','balanced_accuracy_std','f1_mean','f1_std']].sort_values('balanced_accuracy_mean', ascending=False))

,model,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std
9,PCA-RBF-SVM k=8,0.724189,0.012728,0.725437,0.023820
8,PCA-polynomial-SVM k=6,0.722739,0.028987,0.713820,0.042585
10,PCA-polynomial-SVM k=8,0.722501,0.024674,0.715322,0.037994
7,PCA-RBF-SVM k=6,0.721861,0.022261,0.722657,0.026552
6,PCA-polynomial-SVM k=5,0.719375,0.030713,0.717231,0.047667
5,PCA-RBF-SVM k=5,0.718781,0.034106,0.724960,0.036293
0,RBF-SVM all 50 features,0.718542,0.031430,0.726727,0.027127
4,PCA-polynomial-SVM k=4,0.695515,0.040114,0.706122,0.047379
1,Logistic regression,0.690065,0.027182,0.697567,0.024068
3,PCA-RBF-SVM k=4,0.688243,0.040559,0.688017,0.041587


## 3. Ideal quantum-kernel ablation and diagnostics

In [4]:
from src.quantum_kernel import run_ideal_kernel_ablation, print_top_configs
ideal_path = ROOT / 'results/tables/ideal_kernel_ablation.csv'
if RECOMPUTE_IDEAL_ABLATION or not ideal_path.exists():
    ideal = run_ideal_kernel_ablation(ROOT)
else:
    ideal = pd.read_csv(ideal_path)
    print('Loaded cached ideal-kernel ablation:', ideal.shape)
print_top_configs(ideal, ROOT)
display(ideal.sort_values('cv_balanced_accuracy', ascending=False).head(5))

Loaded cached ideal-kernel ablation: (168, 22)

Top 5 ideal quantum-kernel configurations beside the best Prompt 2 classical baseline:
entanglement  reps  n_qubits  bandwidth_s  cv_balanced_accuracy  cv_balanced_accuracy_std  best_C classical_baseline  classical_balanced_accuracy  logical_depth  logical_two_qubit_gates  prepared_depth  prepared_two_qubit_gates  nominal_train_pair_evaluations  nominal_test_vs_train_evaluations
        none     2         6       0.8000                0.7362                    0.0233  1.0000    PCA-RBF-SVM k=8                       0.7242              4                        0             NaN                       NaN                          319600                             160000
      linear     2         6       0.3000                0.7347                    0.0340  1.0000    PCA-RBF-SVM k=8                       0.7242             25                       20             NaN                       NaN                          319600                

,entanglement,reps,n_qubits,bandwidth_s,cv_balanced_accuracy,cv_balanced_accuracy_std,best_C,grid_search_best_cv_balanced_accuracy,offdiag_mean,offdiag_std,...,eigenvalue_max,eigenvalue_negative_count,nominal_train_pair_evaluations,nominal_test_vs_train_evaluations,ideal_statevectors_evaluated,logical_depth,logical_two_qubit_gates,prepared_depth,prepared_two_qubit_gates,prepared_metric_status
39,none,2,6,0.8,0.736174,0.023324,1.0,0.736174,0.752192,0.116798,...,605.538119,0,319600,160000,800,4,0,NaN,NaN,unavailable: ModuleNotFoundError (No module na...
79,linear,2,6,0.3,0.734723,0.034028,1.0,0.734723,0.534277,0.214906,...,443.910745,0,319600,160000,800,25,20,NaN,NaN,unavailable: ModuleNotFoundError (No module na...
19,none,1,6,1.2,0.733703,0.023410,1.0,0.733703,0.649477,0.157565,...,527.261857,0,319600,160000,800,2,0,NaN,NaN,unavailable: ModuleNotFoundError (No module na...
18,none,1,6,0.8,0.730358,0.027658,10.0,0.730358,0.822136,0.091325,...,659.866757,0,319600,160000,800,2,0,NaN,NaN,unavailable: ModuleNotFoundError (No module na...
141,full,1,6,0.2,0.729966,0.035981,10.0,0.729966,0.618263,0.150197,...,501.613087,0,319600,160000,800,29,30,NaN,NaN,unavailable: ModuleNotFoundError (No module na...


## 4. Task A — official Bell benchmark

In [5]:
try:
    from src.task_a import run_task_a
    task_a_result = run_task_a(ROOT)
except Exception as exc:
    print(f'Task A NOT MEASURED in this runtime: {type(exc).__name__}: {exc}')
    print('See results/task_a_run.log and results/task_a_explanation.md. No values are fabricated.')

condition  shots noise_model_version       XX    XX_se        YY    YY_se       ZZ    ZZ_se  fidelity  fidelity_se
    ideal   8192          QFF2026-v2 1.000000 0.000000 -1.000000 0.000000 1.000000 0.000000  1.000000     0.000000
    noisy   8192          QFF2026-v2 0.819092 0.006338 -0.827881 0.006197 0.873535 0.005378  0.880127     0.002592

Saved results/tables/task_a.csv and results/task_a_explanation.md


## 5. Task B — Processor D

In [6]:
try:
    import qiskit, qiskit_aer
except Exception as exc:
    candidate_path = ROOT / 'results/tables/processor_d_candidates.csv'
    if candidate_path.exists(): display(pd.read_csv(candidate_path))
    print(f'Processor D benchmark NOT MEASURED: {type(exc).__name__}: {exc}')
else:
    from src.processor_d import run_sweep
    candidates = run_sweep(ROOT)
    bench_run = subprocess.run([sys.executable, 'benchmark/qff_qml_benchmark.py', 'processor_d_config.json'], cwd=ROOT, capture_output=True, text=True)
    print(bench_run.stdout)
    if bench_run.returncode != 0: print(bench_run.stderr)

                  processor_id topology  n_qubits native_gate_set  directed_edges kernel_entanglement  kernel_reps  kernel_n_qubits  qft3_circuit_depth_before_transpile  qft3_circuit_depth_after_transpile  qft3_two_qubit_gate_count  qft3_swap_overhead  qft3_fidelity_ideal  qft3_fidelity_noisy  kernel_depth_before  kernel_depth_after  kernel_two_qubit_gates_logical  kernel_two_qubit_gates_after_prepare  kernel_swap_overhead qft3_status kernel_transpile_status benchmark_noise_model_version
      procD_line_6_rx_ry_rz_cx   linear         6     rx;ry;rz;cx              10                none            2                6                                    6                                  22                         12                   1                  1.0             0.603585                    4                   6                               0                                     0                     0    measured                measured                    QFF2026-v2
         procD

## 6. Noisy kernel analysis

In [7]:
try:
    import qiskit, qiskit_aer
    from src.noisy_kernel import run_noisy_kernel
    noisy_result = run_noisy_kernel(ROOT, recompute=False)
except Exception as exc:
    print(f'Noisy kernel NOT RUN: {type(exc).__name__}: {exc}')
    print('Noisy metrics must be generated with the unmodified official noise model in the pinned environment.')

Pilot: 20 circuits in 1.58s (0.079s/circuit); estimated budget capacity ~22749 circuit evaluations.
Chosen subset: 100 train, 50 test; 1024 shots; estimated total kernel time 787.3s.
{
  "mean_abs_kernel_error": 0.13463623745105535,
  "ideal_offdiag_mean": 0.7562753582207629,
  "ideal_offdiag_std": 0.11676731679112026,
  "noisy_offdiag_mean": 0.6202791587752525,
  "noisy_offdiag_std": 0.09360257854585811,
  "ideal_alignment": 0.11660652285916494,
  "noisy_alignment": 0.1293921672857417,
  "negative_eigenvalues_before_clip": 7,
  "ideal_balanced_accuracy": 0.7772435897435898,
  "ideal_f1": 0.8,
  "noisy_balanced_accuracy": 0.7475961538461539,
  "noisy_f1": 0.7706422018348624,
  "ideal_best_C": 10.0,
  "noisy_best_C": 1.0,
  "lambda": 0.8236353321918347,
  "c": -0.0008261409721088313,
  "linear_fit_R2": 0.9460002057622144,
  "linear_hypothesis_holds": true,
  "ideal_eigenvalues": [
    5.01797587547115e-06,
    1.1493144001406163e-05,
    1.2760178472260084e-05,
    1.7157129775545004e-0

## 7. Final CV-selected model and test predictions

In [8]:
from src.final_model import run as run_final_model
final_comparison = run_final_model(ROOT)
display(final_comparison)
print('Prediction rows:', len(pd.read_csv(ROOT / 'predictions_q2.csv')))

Leakage-safe CV: ent=none, reps=2, k=6, s=0.8, C=1, BA=0.7362
Leakage-safe CV: ent=linear, reps=2, k=6, s=0.3, C=1, BA=0.7347
Leakage-safe CV: ent=none, reps=1, k=6, s=1.2, C=1, BA=0.7337
Leakage-safe CV: ent=none, reps=1, k=6, s=0.8, C=10, BA=0.7304
Leakage-safe CV: ent=full, reps=1, k=6, s=0.2, C=10, BA=0.7300
Leakage-safe CV: ent=none, reps=2, k=6, s=0.5, C=10, BA=0.7299
Leakage-safe CV: ent=linear, reps=1, k=6, s=0.2, C=10, BA=0.7299
Leakage-safe CV: ent=none, reps=2, k=6, s=1.2, C=1, BA=0.7286
Leakage-safe CV: ent=full, reps=1, k=5, s=0.5, C=1, BA=0.7285
Leakage-safe CV: ent=full, reps=1, k=6, s=0.3, C=1, BA=0.7285
Leakage-safe CV: ent=none, reps=2, k=6, s=0.3, C=10, BA=0.7280
Leakage-safe CV: ent=linear, reps=1, k=6, s=0.3, C=10, BA=0.7273
Selected leakage-aware ideal kernel: none reps 2 k 6 s 0.8 C 1.0
CV BA 0.7362 CV F1 0.7327 predictions 200
                                          model  balanced_accuracy_cv  balanced_accuracy_std    f1_cv   f1_std                           

,model,balanced_accuracy_cv,balanced_accuracy_std,f1_cv,f1_std,status
0,RBF-SVM all 50 features,0.718542,0.031430,0.726727,0.027127,measured; see baseline validation protocol
1,Logistic regression,0.690065,0.027182,0.697567,0.024068,measured; see baseline validation protocol
2,RBF-SVM x01-x06 ablation,0.538952,0.024737,0.541520,0.047806,measured; see baseline validation protocol
3,PCA-RBF-SVM k=4,0.688243,0.040559,0.688017,0.041587,measured; see baseline validation protocol
4,PCA-polynomial-SVM k=4,0.695515,0.040114,0.706122,0.047379,measured; see baseline validation protocol
5,PCA-RBF-SVM k=5,0.718781,0.034106,0.724960,0.036293,measured; see baseline validation protocol
6,PCA-polynomial-SVM k=5,0.719375,0.030713,0.717231,0.047667,measured; see baseline validation protocol
7,PCA-RBF-SVM k=6,0.721861,0.022261,0.722657,0.026552,measured; see baseline validation protocol
8,PCA-polynomial-SVM k=6,0.722739,0.028987,0.713820,0.042585,measured; see baseline validation protocol
9,PCA-RBF-SVM k=8,0.724189,0.012728,0.725437,0.023820,measured; see baseline validation protocol


Prediction rows: 200


## 8. Prediction format validation

In [9]:
import subprocess
check = subprocess.run([sys.executable, 'tools/validate_predictions.py', 'q2', 'predictions_q2.csv'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
if check.returncode != 0: print(check.stderr)
assert check.returncode == 0, 'Prediction validation failed'

OK - predictions file is valid.



## 9. Evidence status

The notebook's successful format-validation cell does not imply that Task A, Task B, or noisy kernel simulations passed. Consult `results.md`, `results/task_a_run.log`, `results/processor_d_sweep_run.log`, and `results/noisy_kernel_run.log` for exact status.